In [1]:
# --- repo bootstrap: make src/ importable and run from repo root (works wherever the kernel starts) ---
import sys, os
from pathlib import Path
_ROOT = Path.cwd()
while _ROOT != _ROOT.parent and not (_ROOT / 'src').is_dir():
    _ROOT = _ROOT.parent
sys.path.insert(0, str(_ROOT / 'src'))
sys.path.insert(0, str(_ROOT / 'schemas'))   # hand-authored reference schemas
os.chdir(_ROOT)

# Aave V3.1 — normalization stage (raw → real units)

    optimized_queries (raw) → API fetch → THIS notebook → transform.ipynb

Builds `query_result_data/batch_<start>_<end>/` — real-unit COPIES of one batch's raw files
(`reserve_config` bps, `reserve_state_rates` RAY, `borrow_repay` token decimals; every other table
copied as is). The flat raw files are never rewritten (E62). `transform` and `validation` read only
the batch folder. Per table, a magnitude guard (`looks_raw_*`) decides whether to scale, so a batch
whose flat files were already rescaled is copied and marked `already_normalized`.

Logic lives in `src/normalize_stage.py` (scaling rules in `normalization.py`).

In [2]:
import pandas as pd
from IPython.display import display
import transform as tf
import normalize_stage as ns

DATA_DIR = "query_result_data"

# Which extraction window to normalize. The menu is built from what each file's ROWS say it
# covers (year+month), not from its name.
display(tf.timelines(DATA_DIR))

,batch,tables,first_date,last_date,select_with
0,2024-04 → 2025-03,9,2024-04-01,2025-03-31,'2024-04'
1,2025-04 → 2026-03,9,2025-04-01,2026-03-31,'2025-04'


In [3]:
# PICK ONE — paste a `select_with` value above; None = the newest timeline.
BATCH = None

In [ ]:
# The run resolves that choice once and can reach ONLY that window's raw files.
run = tf.use_batch(BATCH, source_dir=DATA_DIR)
print(run)
display(run.tables())

In [ ]:
# Raw batch -> query_result_data/batch_<s>_<e>/, rebuilt whole (staging folder swapped in).
# Schema gates, per-table scaling, post-checks and the manifest are all inside normalize_batch.
report = ns.normalize_batch(run)
display(report)
display(run.loaded())

In [ ]:
# ══ TEMPORARY_END_STAMP_SHIFT ═════════════════════════════════════════════════
# TEMPORARY: SQL stamps every bucket at its START (12:00 = 12:00→14:00). The project
# wants END stamps (12:00 = 10:00→12:00). Until the 9 queries are fixed at source,
# shift every time_bucket in the NORMALIZED copies forward by one bucket.
# Delete this cell (and ns.shift_to_end_stamps) when the SQL emits end stamps.
BUCKET_HOURS = 2
display(ns.shift_to_end_stamps(run, BUCKET_HOURS))
# ═══════════════════════════════════════════════════════════════════════════════